# In-depth analysis - class **G4C** in `partition/`

**Mask LUT (segmentation reference):** `[25:74)→GG3`, `[75:174)→GG4`, `[175:)→GG5`; defined as `_MASK_LUT` in the first code cell (gaps in grayscale values 74 and 174 → NC).

**Kernel:** use the interpreter from the project environment (e.g. `prostata_env`) if `pandas`/`openpyxl` fail when reading `.xlsx` files.

Each fold Excel file (`Train.xlsx`, `Test.xlsx`) contains a **main** one-hot label (`NC`, `G3`, `G4`, `G5`) and an additional column **`G4C`**. **G4C is not a fifth parallel class:** it is a **GG4 subtype** - the Gleason 4 **cribriform** pattern versus other Gleason 4 patterns (e.g. acinar/fusionado). In annotation workflows such as the scripts **`TrainCribiform`** (or equivalent pipelines), **only rows already classified as G4 are used**; then each patch is marked as cribriform (**G4C=1**) or non-cribriform Gleason 4 (**G4C=0**). Therefore, G4C implies **G4=1**.

In **4-class segmentation** (NC, GG3, GG4, GG5), patches G4 siguen siendo **GG4** tanto if G4C=1 como if G4C=0; G4C sirve como metadata o for análisis secundario.

Este notebook:
- Recorre **todos** los `.xlsx` bajo `partition/`.
- Cuantifica **G4C** and su co-ocurrencia with `G3`/`G4`/`G5`/`NC`.
- Localiza **slides and pacientes** donde predominan patches G4C.
- Cruza optionalmente with `wsi_labels.xlsx` (Gleason per slide) if está disponible.

**Comprobación:** el crosstab and la celda siguiente verifican que `G4C=1` only appears with `G4=1` (subtype within GG4).

In [ ]:
from __future__ import annotations

from pathlib import Path

import numpy as np
import pandas as pd

try:
    from IPython.display import display
except ImportError:
    display = print  # fuera de Jupyter


def find_project_root() -> Path:
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / "partition").is_dir():
            return p
    raise FileNotFoundError(
        "Not found 'partition/'. Open the notebook from the SICAPv2 project root "
        "or assign BASE = Path(r'...') manually in the next cell."
    )


BASE = find_project_root()
PARTITION = BASE / "partition"
WSI_LABELS = BASE / "wsi_labels.xlsx"

_MASK_LUT = np.zeros(256, dtype=np.int64)
_MASK_LUT[25:74] = 1
_MASK_LUT[75:174] = 2
_MASK_LUT[175:] = 3

print("BASE:", BASE)

In [ ]:
def list_partition_excels() -> list[tuple[Path, str, str]]:
  """Lista (ruta, etiqueta_fold, Train|Test)."""
    out: list[tuple[Path, str, str]] = []
    patterns = [
        ("Validation/*/Train.xlsx", "Train"),
        ("Validation/*/Test.xlsx", "Test"),
        ("Test/Train.xlsx", "Train"),
        ("Test/Test.xlsx", "Test"),
    ]
    for pat, split in patterns:
        for p in sorted(PARTITION.glob(pat)):
            rel = p.relative_to(PARTITION)
            if rel.parts[0] == "Validation" and len(rel.parts) >= 3:
                fold = f"Validation/{rel.parts[1]}"
            elif rel.parts[0] == "Test":
                fold = "Test"
            else:
                fold = "/".join(rel.parts[:-1])
            out.append((p, fold, split))
    return out


excel_files = list_partition_excels()
print(f"Files Excel: {len(excel_files)}")
for path, fold, sp in excel_files:
  print(f" [{fold}] {sp}: {path.name}")

In [ ]:
def load_all_partition_rows() -> pd.DataFrame:
    rows = []
    for path, fold, split in excel_files:
        df = pd.read_excel(path)
        df["_source_file"] = path.name
        df["_partition_relpath"] = str(path.relative_to(BASE))
        df["_fold"] = fold
        df["_split"] = split
        rows.append(df)
    return pd.concat(rows, ignore_index=True)


df_all = load_all_partition_rows()
print("Total rows (image_name may be duplicated across folds):", len(df_all))
print("Columns:", df_all.columns.tolist())

## Where the total number of rows comes from (~51k)

The `DataFrame` **concatenates the 10 Excel files** (each one is a `Train.xlsx` or `Test.xlsx` under a folder). There are **four folds** `Validation/Val1` … `Val4` plus the final `Test/` folder: in cross-validation, **the same patches** (`image_name`) are listed again in each fold with the same annotation. Therefore:

`total rows ≈ rows_per_excel × n_files`, and it is **not** the number of unique dataset patches.

The next table breaks down rows by **relative path** inside the project.

In [ ]:
by_dir = (
    df_all.groupby("_partition_relpath", as_index=False)
    .agg(n_rows=("image_name", "count"))
    .sort_values("_partition_relpath")
    .reset_index(drop=True)
)
by_dir["pct_del_total"] = (100 * by_dir["n_rows"] / len(df_all)).round(2)
display(by_dir)

# Compact view: fold x Train/Test
pivot = df_all.pivot_table(
    index="_fold", columns="_split", values="image_name", aggfunc="count", fill_value=0
)
print("\nRows per fold (row) and split (column):")
display(pivot)

n_unique = df_all["image_name"].nunique()
n_files = df_all["_partition_relpath"].nunique()
print(f"\n--- Summary ---")
print(f"Total rows (sum of the {n_files} Excel): {len(df_all):,}")
print(f"`image_name` unique across the whole concatenation: {n_unique:,}")
print(f"Average rows per file: {len(df_all) / n_files:.0f}")
print(
    f"Repetition: each patch appears ~{len(df_all) / n_unique:.1f} times on average "
    f"(same patches across multiple folds Excel)."
)

## Column schema and one-hot consistency

- **Main class (single):** `NC`, `G3`, `G4`, `G5` — exactly one `1` per row.
- **Subtype only when G4:** `G4C` is an **additional** flag (`0`/`1`) defined **only when the main class is G4** (same criterion as in pipelines such as `TrainCribiform`: the working set for cribriform consists of **G4** rows; then it distinguishes cribriform vs no cribriform).

In [ ]:
df_all[df_all['image_name'] == '18B0001510J_Block_Region_2_0_54_xini_60831_yini_62018.jpg']

In [ ]:
df_all[(df_all['G5'] == 1) & (df_all['_fold'] == 'Validation/Val1')]

In [ ]:
base_cols = [c for c in ["NC", "G3", "G4", "G5"] if c in df_all.columns]
has_g4c = "G4C" in df_all.columns

if not has_g4c:
    raise ValueError("No G4C column found in Excel files; check your dataset version.")

df_all["_main_sum"] = df_all[base_cols].sum(axis=1)
df_all["_main_label"] = df_all[base_cols].idxmax(axis=1)

print("Main-label distribution (NC/G3/G4/G5 by idxmax):")
print(df_all["_main_label"].value_counts())
print()
print("Rows where the sum NC+G3+G4+G5 no es 1:")
bad = df_all["_main_sum"] != 1
print(bad.value_counts())
if bad.any():
    display(df_all.loc[bad, ["image_name", "NC", "G3", "G4", "G5", "G4C", "_main_sum", "_fold"]].head(20))

## Prevalence de **G4C** (subtype within de **G4**) and cruces with la main class

In [ ]:
g4c = df_all["G4C"].fillna(0).astype(int)
n_g4c = int((g4c == 1).sum())
print(f"Patches with G4C=1: {n_g4c:,} ({100 * n_g4c / len(df_all):.2f}% of total loaded rows)")
print(f"Patches with G4C=0: {int((g4c == 0).sum()):,}")

# Cross-tab G4C x main label
ct = pd.crosstab(df_all["_main_label"], g4c, margins=True)
ct.columns = ["G4C=0", "G4C=1", "All"]
print("\nCrosstab: main label (rows) vs G4C (columns)")
print(ct)
print("\nInterpretation: G4C=1 only appears in rows with main label G4 (subtype cribriform within GG4).")
print("NC/G3/G5 do not have G4C=1 in this dataset — consistent with TrainCribiform (only rows G4).")

In [ ]:
# Hierarchy G4 → G4C (as in TrainCribiform: only rows with G4; then cribriform yes/no)
m = g4c == 1
sub = df_all.loc[m, ["NC", "G3", "G4", "G5", "G4C"]].copy()
print("Among rows with G4C=1 (subtype cribriform):")
print(" G4=1:", int((sub["G4"] == 1).sum()), " (expected: all)")
print(" G4=0:", int((sub["G4"] == 0).sum()), " (must be 0 if G4C ⊆ G4)")

only_g4c = df_all.loc[m & (df_all["G4"] == 0)]
print(f"\nRows with G4C=1 but G4=0 (inconsistent): {len(only_g4c)}")
if len(only_g4c):
    display(only_g4c[["image_name", "NC", "G3", "G4", "G5", "G4C", "_fold", "_split"]].head(30))

# Among all rows with main class G4: fracción cribriform (G4C=1) vs no
g4_rows = df_all["_main_label"] == "G4"
n_g4 = int(g4_rows.sum())
n_g4_crib = int((g4_rows & (g4c == 1)).sum())
n_g4_noncrib = n_g4 - n_g4_crib
print(f"\n--- Subdivision of the G4 group (=GG4) ---")
print(f"Rows with main label G4: {n_g4:,}")
print(f" · G4C=1 (GG4 cribriform):   {n_g4_crib:,} ({100 * n_g4_crib / n_g4:.1f}% of the G4)")
print(f" · G4C=0 (GG4 no cribriform): {n_g4_noncrib:,} ({100 * n_g4_noncrib / n_g4:.1f}% of the G4)")
print("\nEsto coincide with la lógica TrainCribiform: el cribriform se anota sobre patches already etiquetados como G4.")

## Per fold and per slide/patient

Los nombres de imagen usuallyn comenzar by un identificador de bloque/slide before de `_Block_`.

In [ ]:
def slide_id_from_image(name: str) -> str:
    s = str(name)
    if "_Block_" in s:
        return s.split("_Block_", 1)[0]
    return s.split("_", 1)[0]


df_all["slide_id"] = df_all["image_name"].map(slide_id_from_image)

g4c_by_fold = (
    df_all.assign(G4C_bin=g4c)
    .groupby(["_fold", "_split"], as_index=False)
    .agg(n=("image_name", "count"), n_g4c=("G4C_bin", "sum"))
)
g4c_by_fold["pct_g4c"] = 100.0 * g4c_by_fold["n_g4c"] / g4c_by_fold["n"]
print("G4C by fold and split:")
display(g4c_by_fold.sort_values(["_fold", "_split"]))

slide_stats = (
    df_all.assign(G4C_bin=g4c)
    .groupby("slide_id")
    .agg(n_patches=("image_name", "count"), n_g4c=("G4C_bin", "sum"))
    .sort_values("n_g4c", ascending=False)
)
slide_stats["pct_g4c"] = 100.0 * slide_stats["n_g4c"] / slide_stats["n_patches"]
print("\nTop 25 slides by número absoluto de patches G4C:")
display(slide_stats.head(25))

## Optional: cross-reference with `wsi_labels.xlsx` (Gleason per slide)

If available, we can check whether slides with many G4C patches show specific WSI-level Gleason patterns.

In [ ]:
if WSI_LABELS.is_file():
    wsi = pd.read_excel(WSI_LABELS)
  print("wsi_labels columns:", wsi.columns.tolist())
    id_col = "slide_id" if "slide_id" in wsi.columns else wsi.columns[0]
    gp = "Gleason_primary" if "Gleason_primary" in wsi.columns else None
    gs = "Gleason_secondary" if "Gleason_secondary" in wsi.columns else None

    top_slides = slide_stats.head(40).index.tolist()
    sub_wsi = wsi[wsi[id_col].astype(str).isin(top_slides)]
    cols = [c for c in [id_col, gp, gs] if c]
  print("\nGleason (WSI) for slides with more G4C patches (sample):")
    display(sub_wsi[cols] if len(cols) > 1 else sub_wsi)

    merged = df_all[["slide_id", "G4C"]].copy()
    merged["G4C"] = merged["G4C"].fillna(0).astype(int)
    slide_g4c_rate = merged.groupby("slide_id")["G4C"].mean().rename("mean_g4c")
    wsi_small = wsi.set_index(id_col)
    if gp and gs:
        wsi_small = wsi_small[[gp, gs]].copy()
        wsi_small["score_str"] = wsi_small[gp].astype(str) + "+" + wsi_small[gs].astype(str)
        joined = wsi_small.join(slide_g4c_rate, how="inner")
    print("\nMedia de G4C by parche vs score Gleason (por slide, only slides in ambos):")
        print(joined.groupby("score_str")["mean_g4c"].agg(["mean", "count"]).sort_values("mean", ascending=False).head(20))
else:
  print(f"Not found {WSI_LABELS}; salta esta sección.")

## Deduplicate by `image_name` (same patch in multiple Excel files)

If the same `image_name` appears in more than one file with the same row, previous counts include it multiple times. Here we keep one row per image name.

In [ ]:
n_before = len(df_all)
df_u = df_all.drop_duplicates(subset=["image_name"], keep="first")
print(f"Unique rows by image_name: {len(df_u)} (before {n_before}, duplicados {n_before - len(df_u)})")

g4c_u = df_u["G4C"].fillna(0).astype(int)
print(f"\nWith deduplication — patches with G4C=1: {int((g4c_u == 1).sum()):,} ({100 * (g4c_u == 1).mean():.2f}%)")

## Interpretive summary

- **G4C (GG4 cribriform)** it is a **GG4 subtype**, not a separate Gleason class from G4. The annotation follows the same idea as in **`TrainCribiform`**: the input set consists of patches with **G4=1**; for those patches, it indicates whether pattern 4 is **cribriform** (`G4C=1`) u otro patrón 4 (`G4C=0`).
- In **4-class segmentation**, todos los `G4=1` son **GG4**; the `G4C` column is used to stratify or analyze cribriform vs no cribriform within GG4.
- The loaded data satisfy **G4C ⊆ G4** (no hay `G4C=1` sin `G4=1`), consistent with that annotation workflow.